In [ ]:
from quspin.basis import spinful_fermion_basis_1d # 用于创建一维1/2自旋费米子链的希尔伯特空间基；
from quspin.operators import hamiltonian # 用于在给定的基（basis）上构建哈密顿量算符(或其他物理观测量)；
import numpy as np 
import matplotlib.pyplot as plt  # 用于结果可视化
from matplotlib.ticker import MultipleLocator # MultipleLocator是matplotlib中的一个刻度定位器类,用来按照指定的倍数来设置坐标轴刻度
import ast
import scipy.linalg as la

In [ ]:
#---------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------########### t-J Model on honeycomb(8格点) ################----------------------------------------
# 参数设置
L = 8 # 格点数
hole_doping =2/8 # 空穴率
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
J1 = 0.2 # 相互作用系数；

#### 计算粒子数（根据空穴浓度）
N_total = int(L * (1 - hole_doping))     # 总电子数；
N_up = int(np.ceil(N_total / 2))    # 上自旋电子数(np.ceil()：向上取整函数)；
N_down = int(np.floor(N_total / 2))  # 下自旋电子数(np.floor()：向下取整函数)；

#### 构建基矢：禁止双占据（每个格点最多一个电子）
basis = spinful_fermion_basis_1d(
    L, 
    Nf=(N_up, N_down), 
    double_occupancy=False
)

### 构建honeycomb的最近邻格点指标i、j的列表
lattice_nn_list = []
# honeycomb竖直方向(即y方向)指标列表的构建
for k in range(0, 5, 4):
    list_y = [(i,(i+1)) for i in range(k, k+3)] # x指标为k时对应的y方向指标列表
    lattice_nn_list.extend(list_y)
    periodic = (k+3,k) # 注：y方向为周期边界
    lattice_nn_list.append(periodic)

# honeycomb水平方向(即x方向)指标列表的构建
list_x = [(0,4), (2,6)] # 注：x方向为开放边界
lattice_nn_list.extend(list_x)
print(f"8格点honeycomb的最近邻格点的列表为：{lattice_nn_list}")

#### 定义site-coupling lists
### 最近邻hopping项
hop_nn_left = [[-t1, i, j] for i,j in lattice_nn_list]  # 直接项：从右向左跃迁项(𝑐†_𝑖,𝜎·𝑐_j,𝜎)；
hop_nn_right = [[t1, i, j] for i,j in lattice_nn_list]  # 厄密共轭项：从左向右跃迁项(𝑐†_j,𝜎·𝑐_𝑖,𝜎= - 𝑐_𝑖,𝜎·c†_j,𝜎)；

### 最近邻相互作用项(自旋相互作用项 + 密度相互作用项)：S_i·S_j-1/4 n_i·n_j = 1/2(S^+_i·S^-_j + S^-_i·S^+_j) + S^z_i·S^z_j - 1/4 n_i·n_j
## J * 1/2(S^+_i·S^-_j + S^-_i·S^+_j)项
int_ss = [[J1/2, i, j, i, j] for i,j in lattice_nn_list]
## J * (S^z_i·S^z_j - 1/4 n_i·n_j) = -J * 1/2(n_i↑·n_j↓ + n_j↑·n_i↓)项
int_nn_ij = [[-J1/2, i, j] for i,j in lattice_nn_list] # n_i↑·n_i↓项；
int_nn_ji = [[-J1/2, j, i] for i,j in lattice_nn_list] # n_j↑·n_i↓项；

# 构建static list
static = [
    ### 最近邻hopping项
    # 上自旋
    ["+-|", hop_nn_left],   # 右向hopping；
    ["-+|", hop_nn_right],  # 左向hopping(厄米共轭)；
    # 下自旋
    ["|+-", hop_nn_left],   # 右向hopping；
    ["|-+", hop_nn_right],  # 左向hopping(厄米共轭)；

    ### 最近邻相互作用项
    ## J * 1/2(S^+_i·S^-_j + S^-_i·S^+_j)项
    ["+-|-+", int_ss],  # S^+_i·S^-_j = 𝑐†_𝑖↑·𝑐_j↑·𝑐_𝑖↓·𝑐†_j↓；
    ["-+|+-", int_ss],  # S^-_i·S^+_j = 𝑐_𝑖↑·𝑐†_j↑·𝑐†_𝑖↓·𝑐_j↓；
    ## J * (S^z_i·S^z_j - 1/4 n_i·n_j) = -J * 1/2(n_i↑·n_j↓ + n_j↑·n_i↓)项
    ["n|n", int_nn_ij],  # n_i↑·n_j↓；
    ["n|n", int_nn_ji]   # n_j↑·n_i↓；
]

dynamic = []  # 无时间依赖项

# 构建哈密顿量
H = hamiltonian(
    static, 
    dynamic, 
    basis=basis, 
    dtype=np.complex128, 
    check_symm=True, 
    check_pcon=True, 
    check_herm=True
)

print('='*80)
print('t-J Model的严格对角化')
print()

## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状s

####### 具体计算
#### 1. 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)


#### 一、 自旋Sz算符平均值
## 计算每个格点的自旋基态平均值<S^z_i>，其中第i格点的Sz_i = 1/2 * (𝑐†_𝑖,↑·𝑐_𝑖,↑ - 𝑐†_𝑖,↓·𝑐_𝑖,↓)
Sz_ED = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_Sz = [
        ["n|", [[0.5, i]]],  # 1/2 * 𝑐†_𝑖,↑·𝑐_𝑖,↑
        ["|n", [[-0.5, i]]] # -1/2 * 𝑐†_𝑖,↓·𝑐_𝑖,↓

    ] 
    dynamic_Sz = [] 
    S_z_i = hamiltonian(static_Sz, dynamic_Sz, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    Sz_i = S_z_i.expt_value(V_gs).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    Sz_ED.append(Sz_i)


#### 二、密度算符平均值
## 计算每个格点的密度基态平均值<n_i>，其中n_i = n_i,↑ + n_i,↓
n_ED = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_n = [
        ["n|", [[1.0, i]]],  # n_i,↑
        ["|n", [[1.0, i]]]   # n_i,↓
    ] 
    dynamic_n = [] 
    ni = hamiltonian(static_n, dynamic_n, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    n_i = ni.expt_value(V_gs).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    n_ED.append(n_i)

#### 三、总自旋平方 <S²>
# 其中S² = S_total·S_total = (∑_i S_i)·(∑_j S_j) = ∑_ij [S^z_i·S^z_j + 1/2 (S^†_i·S^−_j + S^−_i·S^†_j)]
# 定义总自旋平方 S² 的站点耦合列表（包括所有 i,j 对）
pm_coupling = [[0.5, i, j, i, j] for i in range(L) for j in range(L)]  
zz_ij_1 = [[0.25, i, j] for i in range(L) for j in range(L)] 
zz_ij_2 = [[-0.25, i, j] for i in range(L) for j in range(L)] 
zz_ji = [[-0.25, j, i] for i in range(L) for j in range(L)] 

# 静态列表用于 S² 算符
static_S2 = [
    # S^+_i·S^-_j + S^-_i·S^+_j = 𝑐†_𝑖↑·𝑐_j↑·𝑐_𝑖↓·𝑐†_j↓ + 𝑐_𝑖↑·𝑐†_j↑·𝑐†_𝑖↓·𝑐_j↓
    ["+-|-+", pm_coupling],
    ["-+|+-", pm_coupling],
    
    # S^z_i·S^z_j = 1/4(n_i↑·n_j↑ + n_i↓·n_j↓ - n_i↑·n_j↓ - n_j↑·n_i↓)
    ["nn|", zz_ij_1],
    ["|nn", zz_ij_1],
    ["n|n", zz_ij_2],
    ["n|n", zz_ji]
]

# 构建S²算符
S2_operator = hamiltonian(static_S2, [], basis=basis, check_symm=False, check_pcon=False, check_herm=False)

# S²平均值
S2 = S2_operator.expt_value(V_gs).real
# 总自旋量子数S：<S²> = S(S+1)ℏ²，其中ℏ=1，故S满足二次方程：S² + S - <S²> = 0
S = 0.5 * (-1 + np.sqrt(max(0.0, 1 + 4 * S2))) # max(0,...) 防止浮点误差导致负数
# 总自旋平均值与量子数的物理意义：
# 反铁磁海森堡模型（偶数格点）：
# 期望: S = 0, <S²> = 0
# 实际可能: S ≈ 0, <S²> ≈ 0.01（由于边界效应）
# 铁磁系统：
# 期望: S = N/2, <S²> = (N/2)(N/2 + 1)
# 例如：对于N = 8: S = 4, <S²> = 4×5 = 20
# 自旋液体：
# 期望: S ≈ 0, 但可能有量子涨落


#### 将以上测量值打印
print("\n============= 测量结果 ==============")
print(f"\nt-J Model基态能量: {E_gs:.15f}")
print("\n-------- 格点期望值 ---------")
print(f"{'Site':<17} {'<Sz>':<25} {'<n>':<20}")
for i in range(L):
    print(f"{i+1:<10} {Sz_ED[i]:<25.15f} {n_ED[i]:<20.15f}")

print("\n-------- 计算总自旋 <S²> --------")
print(f"总自旋算符期望值 <S²> = {S2:.15f}")
print(f"对应的自旋量子数 S ≈ {S:.15f}")

print("\n---------------------------------------------------------------------------------------------------")
print("<Sz> 与 <n> 的具体取值：")
print(f"\n<Sz> = {Sz_ED}")
print(f"\n<n> = {n_ED}")

In [ ]:
### 输入itensor计算结果
Sz_data = input('请输入itensor计算的单格点 <Sz> 列表: ')
n_data = input('请输入itensor计算的单格点 <n> 列表: ')
Sz_itensor = ast.literal_eval(Sz_data)
n_itensor = ast.literal_eval(n_data)

#### 计算误差
Sz_relative_error = [] # 创建存储每个格点的S_z相对误差
n_relative_error = [] # 创建存储每个格点的n相对误差
Sz_absolute_error = [] # 创建存储每个格点的S_z绝对误差
n_absolute_error = [] # 创建存储每个格点的n绝对误差
for i in range(L):
    Sz_re_err = abs((Sz_ED[i] - Sz_itensor[i])/Sz_ED[i])
    n_re_err = abs((n_ED[i] - n_itensor[i])/n_ED[i])
    Sz_relative_error.append(Sz_re_err)
    n_relative_error.append(n_re_err)
    
    Sz_abs_err = abs(Sz_ED[i] - Sz_itensor[i])
    n_abs_err = abs(n_ED[i] - n_itensor[i])
    Sz_absolute_error.append(Sz_abs_err)
    n_absolute_error.append(n_abs_err)

# 总相对误差(即每个格点相对误差的和)
Sz_total_relative_error = sum(Sz_relative_error)
n_total_relative_error = sum(n_relative_error)
print(f'\n自旋总相对误差: {Sz_total_relative_error:.5e}')
print(f'密度总相对误差: {n_total_relative_error:.5e}')

# 总绝对误差(即每个格点绝对误差的和)
Sz_total_absolute_error = sum(Sz_absolute_error)
n_total_absolute_error = sum(n_absolute_error)
print(f'\n自旋总绝对误差: {Sz_total_absolute_error:.5e}')
print(f'密度总绝对误差: {n_total_absolute_error:.5e}')


#---------------------------------------------------------------------------------------------------------------------
#---------------------------------########### 自旋Sz算符平均值 ################---------------------------------------
print()
print('='*80)
print('每个格点的自旋平均值 <S^z_i>：')
print()

## 可视化S_z平均值
plt.figure(figsize=(10, 5))
plt.plot(range(L), Sz_ED, 'bo-', linewidth=2, markersize=8, label="ED")
plt.plot(range(L), Sz_itensor, 'ys-', linewidth=2, markersize=8, label="Itensor")
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('$\\langle S_i^z \\rangle$', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('Average spin per site $\\langle S_i^z \\rangle$', fontsize=20)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=13)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

plt.show()



#--------------------------------------------------------------------------------------------------------------------
#---------------------------------########### 密度算符平均值 ################----------------------------------------
print()
print('='*80)
print('每个格点的密度平均值 <n_i>：')
print()

## 可视化n平均值
plt.figure(figsize=(10, 5))
plt.plot(range(L), n_ED, 'bo-', linewidth=2, markersize=8, label="ED")
plt.plot(range(L), n_itensor, 'ys-', linewidth=2, markersize=8, label="Itensor")
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('$\\langle n_i \\rangle$', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('Average density per site $\\langle n_i \\rangle$', fontsize=20)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=13)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

plt.show()